# Advanced RFT: Math Reasoning

**TRAINING · ADVANCED / QUOTA-DEPENDENT**

This notebook prepares 100/10 runtime OpenR1-Math-220k subsets from immutable preserved inputs, retaining earlier corrections and three local figure-independent adaptations. An additional labeled runtime adaptation makes training row 1 an answer-only subtask with all three triangle sides; its source proof subpart is not graded. No missing source figure has been reconstructed. It keeps five outcomes separate: grader validity, baseline quality, training completion, deployment availability, and endpoint evaluation.

> **Cloud costs:** This workflow uses paid cloud resources and may incur charges. Upload/training and optional baseline/endpoint inference perform Azure calls. Local data and grader tests do not submit jobs or inference requests.

## 1. Imports, paths, and execution gates

Copy [the blank template](../.env.template) to a demo-local `.env` and set your own values. The training model is `qwen3.6-35b-a3b`, defined by `MODEL` in the preparation cell; the notebook does not read model defaults from `demo.yaml`.

| Setting / value | Meaning and when required |
|---|---|
| `FOUNDRY_PROJECT_ENDPOINT` | Foundry project endpoint; required for every remote section. |
| `MODEL` / `MODEL_VERSION` | Base-model identity `qwen3.6-35b-a3b` / `1`; baseline inference requires this exact deployed identity. Confirm training access, region support, and quota before Section 5. The constant is not proof of the training service's actual base-version selection. |
| `completed_job.fine_tuned_model` | Exact trained artifact ID returned by the matching succeeded job, not an inference deployment name. |
| `FOUNDRY_MATH_BASELINE_DEPLOYMENT` | Existing base-model deployment **name**; required only when `FOUNDRY_RUN_REMOTE_BASELINE=true` (Section 4). |
| `FOUNDRY_MATH_FINE_TUNED_DEPLOYMENT` | Existing fine-tuned deployment **name**; required only when `FOUNDRY_RUN_REMOTE_EVALUATION=true` (Section 9). |
| `FOUNDRY_MATH_FINE_TUNED_MODEL_VERSION` | Exact registered trained-artifact version, independently verified in Foundry; required for Section 9. Never copy it from the candidate deployment or assume the base version. |
| `FOUNDRY_EXISTING_MATH_RFT_JOB_ID` | Optional verified job ID for Section 5 recovery. A blank value resumes the saved state, or starts a new job only when no prior submission is pending. |

Both inference flags default to false when blank/unset. They do **not** disable Section 5 uploads or training. Training can finish without either inference deployment. Before any inference, the notebook requires exact nonempty deployment/model/version identity. Missing or mismatched evidence fails closed. Baseline and endpoint inference may incur charges. The notebook warns once before paid work; it does not add a separate approval gate.

In [ ]:
import importlib.util
import json
import os
import re
import sys
from pathlib import Path

from dotenv import load_dotenv

MODEL = "qwen3.6-35b-a3b"
MODEL_VERSION = "1"

cwd = Path.cwd().resolve()
demo_candidates = [cwd, cwd.parent] if cwd.name == "notebooks" else [cwd]
DEMO_ROOT = next(
    (path for path in demo_candidates if (path / "data/hashes.json").is_file()),
    None,
)
if DEMO_ROOT is None:
    raise FileNotFoundError("Run from the math cookbook or its notebooks directory")
TRACK_ROOT = DEMO_ROOT.parent
sys.path.insert(0, str(TRACK_ROOT / "src"))
sys.path.insert(0, str(DEMO_ROOT))

from rft_common import (
    collect_new_events,
    content_addressed_name,
    env_flag,
    extract_training_metrics,
    load_jsonl,
    require_env,
    retry,
    validate_hash_manifest,
    wait_for_job,
)
from live_validation import submit_or_resume_math_job
from prepare_math_data import prepare_math_data

STATE_PATH = DEMO_ROOT / "outputs/submission-state.json"
_paid_warning_shown = False


def warn_paid_workflow():
    global _paid_warning_shown
    if not _paid_warning_shown:
        print("This workflow uses paid cloud resources and may incur charges.")
        print("Training, baseline inference, and endpoint evaluation may be billed.")
        _paid_warning_shown = True


load_dotenv(DEMO_ROOT / ".env", override=False)

FOUNDRY_RUN_REMOTE_BASELINE = env_flag("FOUNDRY_RUN_REMOTE_BASELINE")
FOUNDRY_RUN_REMOTE_EVALUATION = env_flag("FOUNDRY_RUN_REMOTE_EVALUATION")
print(
    {"remote_baseline": FOUNDRY_RUN_REMOTE_BASELINE, "remote_evaluation": FOUNDRY_RUN_REMOTE_EVALUATION}
)

## 2. Verify preserved sources and prepare the runtime task

The [data notes](../data/README.md) and [manifest](../data/hashes.json) retain source revision, hashes, earlier answer corrections, and the three local adaptations (source training rows 34, 55, 96). Row 14 still requires both lamp counts `(20, 30)`.

The full source row 1 asks for all sides **and** a proof; its stored `9` is incomplete. The [independent derivation](../data/README.md#runtime-row-1-curation-and-independent-proof) gives `(4, 6, 9)` and primitive sides `(m², mn, n²)`. The offline preparation helper leaves all preserved bytes and hashes untouched, explicitly adapts only runtime row 1 to answer-only part (a), and excludes proof part (b) from grading. The split remains 100/10 with no row moved. The other 99 training lines, validation, and five earlier answer corrections are unchanged.

Prepared inputs and a computed source/runtime provenance report live only under ignored `outputs/prepared-data/`. The notebook uses those prepared paths for upload. A differing existing prepared file or old submission equivalence stops rather than overwriting evidence. Hash validation, names, and upload use the same canonical CRLF-to-LF bytes on every platform; JSON string escapes are unchanged.

In [ ]:
prepared_data = prepare_math_data(DEMO_ROOT, DEMO_ROOT / "outputs/prepared-data")
TRAIN_PATH = prepared_data["training_path"]
VALIDATION_PATH = prepared_data["validation_path"]
training_rows = load_jsonl(TRAIN_PATH)
validation_rows = load_jsonl(VALIDATION_PATH)

assert len(training_rows) == 100
assert len(validation_rows) == 10
assert all(set(row) == {"messages", "answer"} for row in training_rows)
assert all(set(row) == {"messages", "answer"} for row in validation_rows)
assert all(len(row["messages"]) == 1 for row in training_rows + validation_rows)
assert all(row["messages"][0]["role"] == "user" for row in training_rows + validation_rows)

train_prompts = {row["messages"][0]["content"] for row in training_rows}
validation_prompts = {row["messages"][0]["content"] for row in validation_rows}
assert train_prompts.isdisjoint(validation_prompts), "Training/validation leakage detected"

print(
    {
        "training_rows": len(training_rows),
        "validation_rows": len(validation_rows),
        "first_answer": training_rows[0]["answer"],
        "runtime_curation": prepared_data["report"]["policy"],
        "split_overlap": 0,
    }
)

Each record contains a problem prompt and a reference final answer, but no assistant reasoning trace. The model explores solution paths; the grader rewards a final `\boxed{...}` answer that matches the reference.

## 3. Load, validate, and attack the boxed-answer grader

In [ ]:
validate_hash_manifest(DEMO_ROOT / "graders/hashes.json", DEMO_ROOT / "graders")

grader_path = DEMO_ROOT / "graders/boxed_math_grader.py"
grader_spec = importlib.util.spec_from_file_location("boxed_math_grader", grader_path)
grader_module = importlib.util.module_from_spec(grader_spec)
grader_spec.loader.exec_module(grader_module)
GRADER_SOURCE = grader_path.read_text(encoding="utf-8")
GRADER = {
    "type": "python",
    "name": "core_rft_boxed_math_answer",
    "source": GRADER_SOURCE,
}

grader_tests = [
    ("exact integer", r"The final answer is \boxed{42}", "42", 1.0),
    ("wrong integer", r"The final answer is \boxed{99}", "42", 0.0),
    ("missing box", "The final answer is 42", "42", 0.0),
    ("last box wins", r"Draft \boxed{0}; final \boxed{42}", "42", 1.0),
    ("final wrong defeats earlier leak", r"Reference \boxed{42}; final \boxed{41}", "42", 0.0),
    ("nested LaTeX", r"Answer: \boxed{\sqrt{2}}", r"\sqrt{2}", 1.0),
    ("unordered answer list", r"Answer: \boxed{3, 2, 1}", "1, 2, 3", 1.0),
    ("spacing normalization", r"Answer: \boxed{x + 1}", "x+1", 1.0),
    ("unclosed box", r"Answer: \boxed{42", "42", 0.0),
    ("reward hack: answer outside final box", r"42, therefore \boxed{0}", "42", 0.0),
]

for name, output, answer, expected in grader_tests:
    actual = grader_module.grade({"output_text": output}, {"answer": answer})
    assert actual == expected, f"{name}: expected {expected}, received {actual}"
    print(f"{name:38} reward={actual}")

print(f"All {len(grader_tests)} grader and reward-hacking tests passed")

The grader intentionally judges only the last balanced `\boxed{...}` expression. Mentioning the reference elsewhere, producing an earlier correct box followed by a wrong final answer, or omitting the box earns zero reward. This limits obvious reward hacking, but it does not prove the reasoning is faithful.

## 4. Define one baseline and endpoint evaluation contract

Optional baseline and endpoint inference use paid chat-completion POSTs. The actual evaluation helper first retrieves the named deployment and verifies an exact nonempty name/model/version match. Baseline identity is `MODEL` / `MODEL_VERSION`; fine-tuned identity is the matching succeeded job's `fine_tuned_model` plus its independently verified registered-artifact version. The low-level solver repeats this guard before each POST. Missing, redacted, or mismatched identity stops without inference; do not use the candidate deployment as its own expected identity.

Each item is submitted once with SDK retries disabled. A timeout may still have incurred a charge; do not assume rerunning this cell is free. Read-only deployment lookups may retry. These checks do not provision a deployment, prove that a mutable deployment cannot later be repointed, or establish the training service's actual base-model version.

In [ ]:
def validate_math_deployment_identity(
    deployment, *, deployment_name, role, completed_job=None,
    expected_job_id=None, artifact_version=None,
):
    # The deployment response is the ACTUAL identity, never its own expectation.
    def exact_text(value):
        return isinstance(value, str) and bool(value) and value == value.strip()

    if not exact_text(deployment_name):
        raise RuntimeError("An exact deployment name is required before inference")
    if role == "baseline":
        expected_model, expected_version = MODEL, MODEL_VERSION
    elif role == "fine-tuned":
        if (
            completed_job is None
            or not exact_text(expected_job_id)
            or getattr(completed_job, "id", None) != expected_job_id
            or getattr(completed_job, "status", None) != "succeeded"
            or getattr(completed_job, "model", None) != MODEL
            or not exact_text(getattr(completed_job, "fine_tuned_model", None))
        ):
            raise RuntimeError("A matching succeeded training job and exact trained artifact are required")
        expected_model = completed_job.fine_tuned_model
        # Jobs expose the artifact ID, not an authoritative serving-version field.
        # Obtain its version independently from that registered artifact in Foundry,
        # never from this deployment lookup or by assuming the base-model version.
        expected_version = artifact_version
    else:
        raise ValueError("Select baseline or fine-tuned deployment verification")
    if not exact_text(expected_model) or not exact_text(expected_version):
        raise RuntimeError(
            "Expected model/version identity is missing. Independently verify the "
            "registered artifact version in Foundry; no inference was sent."
        )
    actual = {
        "name": getattr(deployment, "name", None),
        "model": getattr(deployment, "model_name", None),
        "version": getattr(deployment, "model_version", None),
    }
    expected = {"name": deployment_name, "model": expected_model, "version": expected_version}
    if any(not exact_text(value) for value in actual.values()) or actual != expected:
        raise RuntimeError(
            "Deployment model/version identity is missing or mismatched; no inference was sent. "
            "Do not substitute the deployment's observed model/version as the expected identity."
        )
    return {**expected, "role": role}


def solve_math(
    client, deployment, row, *, deployment_name, role,
    completed_job=None, expected_job_id=None, artifact_version=None,
):
    identity = validate_math_deployment_identity(
        deployment, deployment_name=deployment_name, role=role,
        completed_job=completed_job, expected_job_id=expected_job_id,
        artifact_version=artifact_version,
    )
    warn_paid_workflow()
    # Paid, non-idempotent POST: do not retry here or in the SDK.
    response = client.with_options(max_retries=0).chat.completions.create(
        model=identity["name"],
        messages=row["messages"],
        max_completion_tokens=20_000,
    )
    output = response.choices[0].message.content or ""
    reward = grader_module.grade({"output_text": output}, {"answer": row["answer"]})
    return {
        "reward": reward,
        "correct": reward == 1.0,
        "word_count": len(output.split()),
        "output": output,
    }


def evaluate_math_deployment(
    project, deployment_name, rows, limit=10, *, role,
    completed_job=None, expected_job_id=None, artifact_version=None,
):
    deployment = retry(
        lambda: project.deployments.get(deployment_name),
        description="retrieve math deployment identity before inference",
    )
    identity = validate_math_deployment_identity(
        deployment, deployment_name=deployment_name, role=role,
        completed_job=completed_job, expected_job_id=expected_job_id,
        artifact_version=artifact_version,
    )
    client = project.get_openai_client()
    results = [
        solve_math(
            client, deployment, row, deployment_name=deployment_name, role=role,
            completed_job=completed_job, expected_job_id=expected_job_id,
            artifact_version=artifact_version,
        )
        for row in rows[:limit]
    ]
    return {
        "deployment": identity["name"],
        "verified_identity": identity,
        "correct": sum(result["correct"] for result in results),
        "total": len(results),
        "mean_reward": sum(result["reward"] for result in results) / len(results),
        "results": results,
    }


baseline_result = None
if FOUNDRY_RUN_REMOTE_BASELINE:
    values = require_env(
        "FOUNDRY_PROJECT_ENDPOINT",
        "FOUNDRY_MATH_BASELINE_DEPLOYMENT",
    )
    from azure.ai.projects import AIProjectClient
    from azure.identity import DefaultAzureCredential

    baseline_project = AIProjectClient(
        endpoint=values["FOUNDRY_PROJECT_ENDPOINT"],
        credential=DefaultAzureCredential(),
    )
    baseline_result = evaluate_math_deployment(
        baseline_project,
        values["FOUNDRY_MATH_BASELINE_DEPLOYMENT"],
        validation_rows,
        limit=10,
        role="baseline",
    )
    print(
        {
            key: baseline_result[key]
            for key in ("deployment", "verified_identity", "correct", "total", "mean_reward")
        }
    )
else:
    print("Skipped remote baseline; offline grader validation is complete.")

## 5. Prepare content-addressed files and the RFT job

Running the next upload/training cell submits or resumes a paid job. The ignored `outputs/submission-state.json` binds the project endpoint, model, grader/method, training type, suffix, and canonical input hashes/names. It records top-level `training_file_id`, `validation_file_id`, and `job_id` for [post-training cancellation and cleanup](../../../POST_TRAINING.md). Keep it on durable local storage; do not commit it or erase it merely to rerun.

The helper atomically flushes state **before each upload/job POST** and saves each returned ID **before processing or another request**. Upload and job POSTs have SDK retries disabled. A submission lock prevents concurrent notebook writers; after a process crash, reconcile operations and confirm no writer is active before manually removing a retained lock.

**Resume after kernel loss:** restart the kernel, keep the same project/model/data/settings, then run Sections 1–5. The saved job and file IDs are retrieved and verified without new POSTs. Failed, cancelled, or mismatched jobs stop; no automatic replacement is submitted.

**Ambiguous upload:** exactly one canonical-name match with verified downloaded content can recover the file. No visible match does not prove failure, so it stops instead of repeating the POST.

**Ambiguous job submission:** check Foundry. Set `FOUNDRY_EXISTING_MATH_RFT_JOB_ID` to the verified job ID, then restart and resume; the helper records and validates its file IDs before reuse. If Foundry confirms no resource was created, retain the evidence and intentionally clear only the corresponding pending marker before a new attempt. Never blindly delete state or clear a job ID to bypass reconciliation.

In [ ]:
MATH_METHOD = {
    "type": "reinforcement",
    "reinforcement": {
        "grader": GRADER,
        "hyperparameters": {
            "n_epochs": 2,
            "batch_size": 128,
            "learning_rate_multiplier": 2.0,
            "eval_interval": 25,
            "eval_samples": 2,
            "reasoning_effort": "medium",
        },
    },
}

remote_names = {
    "training": content_addressed_name("core-rft-openr1-math-training", TRAIN_PATH),
    "validation": content_addressed_name("core-rft-openr1-math-validation", VALIDATION_PATH),
}
remote_names

In [ ]:
fine_tuning_job = None
training_client = None
project_client = None

values = require_env("FOUNDRY_PROJECT_ENDPOINT")
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential

project_client = AIProjectClient(
    endpoint=values["FOUNDRY_PROJECT_ENDPOINT"],
    credential=DefaultAzureCredential(),
)
training_client = project_client.get_openai_client()
warn_paid_workflow()
fine_tuning_job = submit_or_resume_math_job(
    training_client,
    state_path=STATE_PATH,
    project_endpoint=values["FOUNDRY_PROJECT_ENDPOINT"],
    model=MODEL,
    method=MATH_METHOD,
    training_path=TRAIN_PATH,
    validation_path=VALIDATION_PATH,
    existing_job_id=os.getenv("FOUNDRY_EXISTING_MATH_RFT_JOB_ID") or None,
)
print("Math RFT status:", fine_tuning_job.status)
print("Recovery IDs are saved locally under outputs; retain that state for cleanup.")

## 6. Monitor with a timeout and collect real reward/loss events

The six-hour timeout stops local monitoring, **not** the remote job. Check its status in Foundry before retrying. With the kernel intact, rerun only this monitoring cell; after kernel loss, recover the existing job as described in Section 5. Cancellation is a separate, explicit operation; see [post-training guidance](../../../POST_TRAINING.md).

In [ ]:
training_outcome = None
event_metrics = {"reward": [], "loss": []}

completed_job = wait_for_job(
    training_client,
    fine_tuning_job.id,
    timeout_seconds=21_600,
    poll_seconds=60,
)
seen_ids = set()
events = collect_new_events(training_client, fine_tuning_job.id, seen_ids)
event_messages = [event.message for event in events]
event_metrics = extract_training_metrics(event_messages)
training_outcome = {
    "status": completed_job.status,
    "registered_model": completed_job.fine_tuned_model,
    "eval_id": completed_job.eval,
    "event_metrics": event_metrics,
}
print(training_outcome)


![Representative validation reward](../assets/charts/representative-validation-reward.svg)

The committed curve contains the two validation means from the historical run using pre-correction inputs: `0.05` and `0.20`. These are not results for the current corrected data. A current run should use service events and evaluation outputs from the previous cell; never substitute this historical curve for current evidence.

## 7. Retrieve validation rewards and inspect a successful reasoning sample

In [ ]:
validation_outcome = None
completed_job = training_client.fine_tuning.jobs.retrieve(fine_tuning_job.id)
eval_id = completed_job.eval
eval_runs = list(training_client.evals.runs.list(eval_id, limit=100).data)
completed_runs = sorted(
    (run for run in eval_runs if run.status == "completed"),
    key=lambda run: run.created_at,
)
if not completed_runs:
    raise RuntimeError(f"No completed validation runs found for {eval_id}")

reward_history = []
selected_item = None
for eval_run in completed_runs:
    output_items = list(
        training_client.evals.runs.output_items.list(
            run_id=eval_run.id,
            eval_id=eval_id,
            limit=100,
        ).data
    )
    scores = [
        float(item.results[0].score)
        for item in output_items
        if item.results
    ]
    if not scores:
        raise RuntimeError(f"Evaluation run {eval_run.id} returned no scores")
    reward_history.append(sum(scores) / len(scores))
    if eval_run is completed_runs[-1]:
        selected_item = next(
            (
                item
                for item in output_items
                if item.results and float(item.results[0].score) == 1.0
            ),
            None,
        )

if selected_item is None:
    raise AssertionError("Final validation run has no correct output")
sample = selected_item.datasource_item["sample"]["output_text"]
reference = str(selected_item.datasource_item["answer"])
if grader_module.grade({"output_text": sample}, {"answer": reference}) != 1.0:
    raise AssertionError("Selected validation output does not pass the local grader")
if len(sample.split()) < 40:
    raise AssertionError("Selected output lacks a substantive reasoning trace")

validation_outcome = {
    "reward_history": reward_history,
    "correct_sample_words": len(sample.split()),
    "final_run": completed_runs[-1].id,
}
print(validation_outcome)


## 8. Check deployment availability separately

In [ ]:
deployment_outcome = None
artifact_id = getattr(completed_job, "fine_tuned_model", None)
if (
    completed_job.status != "succeeded"
    or completed_job.id != fine_tuning_job.id
    or completed_job.model != MODEL
    or not isinstance(artifact_id, str) or not artifact_id.strip()
):
    raise RuntimeError("A matching succeeded job and trained artifact are required for deployment discovery")
matching = [
    item for item in project_client.deployments.list(model_name=artifact_id)
    if getattr(item, "model_name", None) == artifact_id
]
if matching:
    deployment_outcome = {
        "status": "candidates_only_identity_not_yet_verified",
        "model": artifact_id,
        "candidates": [
            {"name": item.name, "model": item.model_name,
             "version": getattr(item, "model_version", None)}
            for item in matching
        ],
    }
else:
    deployment_outcome = {
        "status": "not_available",
        "model": artifact_id,
        "next_step": (
            "Confirm supported model/region and sufficient deployment quota, "
            "then create the deployment in Foundry."
        ),
    }
print(deployment_outcome)

**Manual deployment handoff:** Section 8 only lists candidates; no section creates a deployment or treats a list filter as verified identity. A matching succeeded job registers `completed_job.fine_tuned_model`.

1. In Foundry, inspect that exact trained artifact and independently record its registered version. If deployment is needed, use your approved provisioning workflow after checking support, permissions, quota, capacity, and serving cost; this notebook does not provision it.
2. Set `FOUNDRY_MATH_FINE_TUNED_DEPLOYMENT` to the **deployment name**, and `FOUNDRY_MATH_FINE_TUNED_MODEL_VERSION` to the independently verified **artifact version**. Never copy the expected version from the candidate deployment or assume it is the base version. Section 9 re-retrieves the matching job and fails closed unless deployment name, exact artifact, and version all agree.
3. Enable `FOUNDRY_RUN_REMOTE_EVALUATION` and run Section 9. If configuration was already loaded, update the relevant `os.environ` entries and the kernel flag; editing `.env` does not refresh cached settings. Do not use **Run All** to enable evaluation: Section 5 performs paid submission/resume work.

Without verified deployment identity, leave endpoint evaluation disabled. Training and service validation remain distinct from endpoint quality; missing/redacted identity is not permission to guess. See [serving guidance](../../../POST_TRAINING.md).

## 9. Evaluate a deployed endpoint with the same validation contract

In [ ]:
endpoint_evaluation = None
if FOUNDRY_RUN_REMOTE_EVALUATION:
    values = require_env(
        "FOUNDRY_PROJECT_ENDPOINT",
        "FOUNDRY_MATH_FINE_TUNED_DEPLOYMENT",
        "FOUNDRY_MATH_FINE_TUNED_MODEL_VERSION",
    )
    if training_client is None or fine_tuning_job is None:
        raise RuntimeError("Resume and verify the saved training job before endpoint evaluation")
    completed_job = retry(
        lambda: training_client.fine_tuning.jobs.retrieve(fine_tuning_job.id),
        description="retrieve completed math job before endpoint evaluation",
    )
    # The client is bound to the project already used to submit/resume this job.
    # Do not create a second client from a possibly changed endpoint/environment.
    endpoint_evaluation = evaluate_math_deployment(
        project_client,
        values["FOUNDRY_MATH_FINE_TUNED_DEPLOYMENT"],
        validation_rows,
        limit=10,
        role="fine-tuned",
        completed_job=completed_job,
        expected_job_id=fine_tuning_job.id,
        artifact_version=values["FOUNDRY_MATH_FINE_TUNED_MODEL_VERSION"],
    )
    print(
        {
            key: endpoint_evaluation[key]
            for key in ("deployment", "verified_identity", "correct", "total", "mean_reward")
        }
    )
else:
    print("Skipped endpoint evaluation; it requires an identity-verified deployment.")

## 10. Outcome distinctions, conclusion, and next step

In [ ]:
outcome_summary = {
    "grader_validation": "passed offline",
    "baseline": baseline_result or "not run",
    "training": training_outcome or "not submitted in this run",
    "validation": validation_outcome or "representative evidence shown",
    "deployment": deployment_outcome or "not checked in this run",
    "endpoint_evaluation": endpoint_evaluation or "not run",
}
outcome_summary

### Conclusion

The advanced-math run demonstrates that training completion, rising validation reward, deployment availability, and endpoint quality are different claims. The boxed-answer grader gives a useful automatic signal, but it verifies the final expression rather than the faithfulness of every reasoning step.

**Next step:** secure supported deployment capacity, add an untouched test split, evaluate the base and fine-tuned deployments on that independent set, and manually review correct and incorrect chains for reasoning quality—not just final-answer reward.

*Variability footnote:* the `0.05 → 0.20` validation rewards and 257-word correct response come from one historical run using pre-correction inputs, not the current corrected data. Model availability, quota, service updates, grader behavior, data composition, and training stochasticity can materially change results.

> **Variability note:** Actual results may vary by model version, data, configuration, region availability, and service conditions.